In [0]:
def create_tables():
    spark.sql("DROP TABLE IF EXISTS data_ai_dbx_learning_workspace.default.accounts")
    spark.sql("DROP TABLE IF EXISTS data_ai_dbx_learning_workspace.default.transfer_log")

    spark.sql("""
        CREATE TABLE data_ai_dbx_learning_workspace.default.accounts (
            account_id BIGINT,
            owner      STRING,
            balance    DECIMAL(12, 2)
            ) TBLPROPERTIES ('delta.feature.catalogManaged' = 'supported');
     """)

    spark.sql("""
        CREATE TABLE data_ai_dbx_learning_workspace.default.transfer_log (
            from_account BIGINT,
            to_account   BIGINT,
            amount       DECIMAL(12, 2),
            logged_at    TIMESTAMP
        ) TBLPROPERTIES ('delta.feature.catalogManaged' = 'supported');
     """)
    
    spark.sql("""
        ALTER TABLE data_ai_dbx_learning_workspace.default.accounts
            ADD CONSTRAINT positive_balance CHECK (balance >= 0);
    """)

    spark.sql("""
        INSERT INTO data_ai_dbx_learning_workspace.default.accounts VALUES
            (1, 'Ada',   500.00),
            (2, 'Grace', 125.00);
    """)

create_tables()

In [0]:
# This operation should be acceptable as 'Ada' has enough balance to transfer to 'Grace'
from_id = 1
to_id = 2
amount = 100.00

spark.sql(f"INSERT INTO data_ai_dbx_learning_workspace.default.transfer_log VALUES ({from_id}, {to_id}, {amount}, current_timestamp())")
spark.sql(f"UPDATE data_ai_dbx_learning_workspace.default.accounts SET balance = balance + {amount} WHERE account_id = {to_id}")
spark.sql(f"UPDATE data_ai_dbx_learning_workspace.default.accounts SET balance = balance - {amount} WHERE account_id = {from_id}")

spark.table('data_ai_dbx_learning_workspace.default.accounts').show()
spark.table('data_ai_dbx_learning_workspace.default.transfer_log').show()

In [0]:
# This operation should fail as 'Grace' doesn't have enough balance,
# however the transaction is still recorded and 'Ada's balance is updated
from_id = 2
to_id = 1
amount = 500

try:
    spark.sql(f"INSERT INTO data_ai_dbx_learning_workspace.default.transfer_log VALUES ({from_id}, {to_id}, {amount}, current_timestamp())")
    spark.sql(f"UPDATE data_ai_dbx_learning_workspace.default.accounts SET balance = balance + {amount} WHERE account_id = {to_id}")
    spark.sql(f"UPDATE data_ai_dbx_learning_workspace.default.accounts SET balance = balance - {amount} WHERE account_id = {from_id}")
except Exception as e:
    if "DELTA_VIOLATE_CONSTRAINT_WITH_VALUES" in str(e):
        print(f"Constraint Raised DELTA_VIOLATE_CONSTRAINT_WITH_VALUES Error")

spark.table('data_ai_dbx_learning_workspace.default.accounts').show()
spark.table('data_ai_dbx_learning_workspace.default.transfer_log').show()

In [0]:
create_tables()

# Using ATOMIC, all 3 operations are executed atomically, thus the transaction is rolled back
from_id = 2
to_id = 1
amount = 500

try:
    spark.sql(f"""
        BEGIN ATOMIC
            INSERT INTO data_ai_dbx_learning_workspace.default.transfer_log VALUES ({from_id}, {to_id}, {amount}, current_timestamp());
            UPDATE data_ai_dbx_learning_workspace.default.accounts SET balance = balance + {amount} WHERE account_id = {to_id};
            UPDATE data_ai_dbx_learning_workspace.default.accounts SET balance = balance - {amount} WHERE account_id = {from_id};
        END;
    """)
except Exception as e:
    if "DELTA_VIOLATE_CONSTRAINT_WITH_VALUES" in str(e):
        print(f"Constraint Raised DELTA_VIOLATE_CONSTRAINT_WITH_VALUES Error")

spark.table('data_ai_dbx_learning_workspace.default.accounts').show()
spark.table('data_ai_dbx_learning_workspace.default.transfer_log').show()

In [0]:
spark.sql("DROP TABLE IF EXISTS data_ai_dbx_learning_workspace.default.accounts")
spark.sql("DROP TABLE IF EXISTS data_ai_dbx_learning_workspace.default.transfer_log")